In [5]:
import polars as pl
from pathlib import Path
import numpy as np
import datetime as dt

In [8]:
def load_symbol_folder(folder_path, start_date=None, end_date=None):
    def to_date(x):
        if x is None:
            return None
        if isinstance(x, dt.date):
            return x
        if isinstance(x, str):
            return dt.date.fromisoformat(x)
        raise TypeError(
            "start_date/end_date must be datetime.date, 'YYYY-MM-DD' string, or None"
        )

    start_date = to_date(start_date)
    end_date = to_date(end_date)

    files = sorted(Path(folder_path).glob("*.parquet"))

    def file_date(p: Path) -> dt.date:
        # expects: <symbol>_YYYY-MM-DD.parquet
        ds = p.stem.rsplit("_", 1)[-1]
        return dt.date.fromisoformat(ds)

    selected = []
    for f in files:
        d = file_date(f)
        if start_date is not None and d < start_date:
            continue
        if end_date is not None and d > end_date:
            continue
        selected.append(f)

    first_schema = pl.read_parquet_schema(selected[0])
    n_cols = len(first_schema)

    valid_files = []
    for f in selected:
        schema = pl.read_parquet_schema(f)
        if len(schema) == n_cols:
            valid_files.append(f)

    df = pl.concat([pl.read_parquet(f) for f in valid_files], how="vertical").sort(
        "ts_event"
    )
    return df


start_date = "2024-07-22"
end_date = "2024-09-15"
cxw_data = load_symbol_folder("CXW_data", start_date=start_date, end_date=end_date)
geo_data = load_symbol_folder("GEO_data", start_date=start_date, end_date=end_date)

In [9]:
def filter_market_hours(df):
    return (
        df
        .with_columns(pl.col("ts_event").dt.convert_time_zone("US/Eastern"))
        .filter(
            (pl.col("ts_event").dt.hour() > 10) |
            ((pl.col("ts_event").dt.hour() == 10) & (pl.col("ts_event").dt.minute() >= 30))
        )
        .filter(pl.col("ts_event").dt.hour() < 15)
    )

def add_midprice(df):
    return df.with_columns(
        ((pl.col("bid_px_00") + pl.col("ask_px_00")) / 2).alias("mid")
    )

def resample_1min(df):
    return (
        df
        .group_by_dynamic(
            "ts_event",
            every="1m",
            closed="right"
        )
        .agg(pl.col("mid").last())
        .drop_nulls()
    )

In [10]:
geo = resample_1min(add_midprice(filter_market_hours(geo_data)))
cxw = resample_1min(add_midprice(filter_market_hours(cxw_data)))

merged = geo.join(cxw, on="ts_event", how="inner", suffix="_cxw")

pdf = (
    merged.select(["ts_event", "mid", "mid_cxw"])
    .rename({"mid": "mid_geo"})
    .with_columns(
        pl.col("ts_event")
        .dt.convert_time_zone("US/Eastern")
        .dt.replace_time_zone(None)
        .alias("ts_event")
    )
    .to_pandas()
    .set_index("ts_event")
)

In [12]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

Y = pdf[["mid_geo", "mid_cxw"]].dropna().values

jres = coint_johansen(Y, det_order=0, k_ar_diff=1)

trace_stat_r0 = jres.lr1[0]
trace_cv95_r0 = jres.cvt[0, 1]

print("Johansen TRACE test (5% level)")
print(f"Test statistic (r <= 0): {trace_stat_r0:.4f}")
print(f"95% critical value     : {trace_cv95_r0:.4f}")

if trace_stat_r0 > trace_cv95_r0:
    print("\nConclusion:")
    print("Reject H0 of no cointegration (r = 0).")
    print("There IS evidence of a cointegration relationship between GEO and CXW.")
else:
    print("\nConclusion:")
    print("Fail to reject H0 of no cointegration (r = 0).")
    print("There is NO statistical evidence of cointegration between GEO and CXW.")

Johansen TRACE test (5% level)
Test statistic (r <= 0): 16.2093
95% critical value     : 15.4943

Conclusion:
Reject H0 of no cointegration (r = 0).
There IS evidence of a cointegration relationship between GEO and CXW.


In [14]:
from statsmodels.tsa.stattools import adfuller

adf_geo = adfuller(pdf["mid_geo"])
adf_cxw = adfuller(pdf["mid_cxw"])
pvalue_geo = adf_geo[1]
pvalue_cxw = adf_cxw[1]
print("p-value GEO: ", pvalue_geo)
print("p-value CXW: ", pvalue_cxw)

signif = 0.05
if pvalue_geo < signif:
    print(f"Decision @ {signif:.0%}: Reject unit root → GEO likely STATIONARY.")
else:
    print(f"Decision @ {signif:.0%}: Fail to reject unit root → GEO likely NON-stationary.")

if pvalue_cxw < signif:
    print(f"Decision @ {signif:.0%}: Reject unit root → CXW likely STATIONARY.")
else:
    print(f"Decision @ {signif:.0%}: Fail to reject unit root → CXW likely NON-stationary.")

p-value GEO:  0.18208141949727213
p-value CXW:  0.2537053319340091
Decision @ 5%: Fail to reject unit root → GEO likely NON-stationary.
Decision @ 5%: Fail to reject unit root → CXW likely NON-stationary.


In [17]:
import statsmodels.api as sm

X = sm.add_constant(pdf["mid_cxw"])
model = sm.OLS(pdf["mid_geo"], X).fit()

beta = model.params["mid_cxw"]
alpha = model.params["const"]

spread = pdf["mid_geo"] - alpha - beta * pdf["mid_cxw"]

adf_spread = adfuller(spread)
pvalue_spread = adf_spread[1]
print("p-value spread: ", pvalue_spread)

if pvalue_spread < signif:
    print(f"Decision @ {signif:.0%}: Reject unit root → Spread likely STATIONARY.")
else:
    print(
        f"Decision @ {signif:.0%}: Fail to reject unit root → Spread likely NON-stationary."
    )

p-value spread:  0.04371554547434296
Decision @ 5%: Reject unit root → Spread likely STATIONARY.


In [115]:
spread_lag = spread.shift(1).dropna()
spread_curr = spread.loc[spread_lag.index]

spread_reg_res = sm.OLS(spread_curr, spread_lag).fit()
rho = spread_reg_res.params.iloc[0]
print("Spread autocorrelation: ", rho)

Spread autocorrelation:  0.9989057711592483


In [ ]:
from statsmodels.tsa.api import VAR

S = pdf[["mid_cxw", "mid"]].dropna()
dt = 1.0 

dS = S.diff().dropna()
var_res = VAR(dS).fit(1)

A = var_res.params.iloc[0].values
B = var_res.coefs[0]

I = np.eye(B.shape[0])
kappa = (I - B) / dt 

theta = np.linalg.solve(kappa, A * dt) 

eigvals, eigvecs = np.linalg.eig(kappa) 

idx = np.argmax(np.real(eigvals))
v = np.real(eigvecs[:, idx]) 

alpha = v / (np.abs(v).max() + 1e-12)

spread = S.values @ alpha  
spread = pl.DataFrame({"ts_event": S.index, "spread_best": spread})

print("VAR(1) on ΔS_t results")
print("A (intercept):", A)
print("B (lag-1 matrix):\n", B)
print("\nDerived kappa = (I - B)/dt:\n", kappa)
print("Derived theta = kappa^{-1} A dt:", theta)

print("\nEigenvalues of kappa:", eigvals)
print("Chosen eigenvalue (fastest MR):", eigvals[idx])

print("\nBest linear combination (up to scale/sign):")
print(f"  spread = {alpha[0]: .6f} * CXW  + {alpha[1]: .6f} * GEO")

if np.abs(alpha[1]) > 1e-12:
    c = alpha[0] / alpha[1]
    print("\nSame combo normalized to GEO coefficient = 1:")
    print(f"  spread = {c: .6f} * CXW  + 1.000000 * GEO")

spread.head()

VAR(1) on ΔS_t results
A (intercept): [-3.68485188e-05 -8.24895786e-05]
B (lag-1 matrix):
 [[-0.04997228  0.0511725 ]
 [ 0.07314717 -0.06707727]]

Derived kappa = (I - B)/dt:
 [[ 1.04997228 -0.0511725 ]
 [-0.07314717  1.06707727]]
Derived theta = kappa^{-1} A dt: [-3.89925992e-05 -7.99771292e-05]

Eigenvalues of kappa: [0.99674882 1.12030073]
Chosen eigenvalue (fastest MR): 1.1203007265689133

Best linear combination (up to scale/sign):
  spread =  0.727622 * CXW  + -1.000000 * GEO

Same combo normalized to GEO coefficient = 1:
  spread = -0.727622 * CXW  + 1.000000 * GEO


c:\Users\Laurent Liao\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


ts_event,spread_best
datetime[ns],f64
2024-07-22 10:30:00,-5.680849
2024-07-22 10:31:00,-5.695849
2024-07-22 10:32:00,-5.700849
2024-07-22 10:33:00,-5.694935
2024-07-22 10:34:00,-5.679021
